# VehiAlpes — Capa Gold: hechos

MINE-4214 · Taller 1 · Punto 2

## Objetivo

Materializar los seis hechos de la matriz de bus: tres transaccionales que se
alimentan de silver, y tres derivados que se construyen a partir de los primeros.

## La decisión central de esta capa

**Separar `transacciones` en tres hechos en lugar de cargarla como una sola tabla.**

La fuente entrega los tres procesos mezclados. Cargarla tal cual produciría un
hecho con tres defectos simultáneos que los criterios de calidad del curso
descartan:

| Defecto | Evidencia en los datos |
|---|---|
| Granos mezclados | Una compra, un contrato de alquiler y una venta no son el mismo evento |
| Atributos que no aplican | `nombre_proveedor` nulo en 90% de las filas; `kms_recorridos` y `valor_seguro` solo en alquileres |
| Medidas ajenas al proceso | `valor_total` significa precio de compra, ingreso de alquiler o precio de venta según el tipo |

El principio de fondo es que un modelo dimensional representa un **proceso de
negocio**, no una fuente de datos. Tres procesos, tres hechos.

## Decisión: el join del vehículo va por rango de vigencia

Todos los hechos resuelven `sk_vehiculo` buscando la versión cuya vigencia contiene
la fecha del evento, no la versión actual. Esto es lo que hace que la tarifa
histórica sea correcta y es la razón de ser del SCD tipo 2.

In [0]:
from pyspark.sql import functions as F, Window as W

CATALOGO = "vehialpes"
SK_NO_APLICA = -1

sv = spark.table(f"{CATALOGO}.silver.transacciones")
dim_vehiculo = spark.table(f"{CATALOGO}.gold.dim_vehiculo")
dim_cliente = spark.table(f"{CATALOGO}.gold.dim_cliente")
dim_sucursal = spark.table(f"{CATALOGO}.gold.dim_sucursal")
dim_proveedor = spark.table(f"{CATALOGO}.gold.dim_proveedor")
dim_pago = spark.table(f"{CATALOGO}.gold.dim_pago")

## Funciones de resolución de llaves

Se factorizan porque los seis hechos las reutilizan. Cada una aplica `coalesce`
contra el centinela para garantizar que **ninguna FK quede nula**.

In [0]:
def sk_vehiculo_vigente(df, col_fecha="fecha_inicio"):
    """Resuelve la versión del vehículo vigente en la fecha del evento."""
    v = dim_vehiculo.filter(F.col("sk_vehiculo") > 0).select(
        F.col("sk_vehiculo"), F.col("placa").alias("_p"),
        F.col("fe_vig_ini").alias("_ini"), F.col("fe_vig_fin").alias("_fin"))
    return (df.join(v,
            (F.col("placa") == F.col("_p")) &
            (F.col(col_fecha) >= F.col("_ini")) &
            (F.col(col_fecha) <= F.col("_fin")), "left")
        .withColumn("sk_vehiculo", F.coalesce("sk_vehiculo", F.lit(SK_NO_APLICA)))
        .drop("_p", "_ini", "_fin"))

def sk_simple(df, dim, col_natural, col_dim, sk):
    d = dim.filter(F.col(sk) > 0).select(F.col(sk), F.col(col_dim).alias("_k"))
    return (df.join(d, F.col(col_natural) == F.col("_k"), "left")
        .withColumn(sk, F.coalesce(sk, F.lit(SK_NO_APLICA))).drop("_k"))

def sk_pago(df):
    d = dim_pago.select(
        "sk_pago",
        F.col("metodo_pago").alias("_m"),
        F.col("ind_valor_inconsistente").alias("_a"),
        F.col("ind_fecha_reformateada").alias("_b"),
        F.col("ind_registro_deduplicado").alias("_c"))
    return (df.join(d,
            (F.col("metodo_pago") == F.col("_m")) &
            (F.coalesce("ind_valor_inconsistente", F.lit(False)) == F.col("_a")) &
            (F.coalesce("ind_fecha_reformateada", F.lit(False)) == F.col("_b")) &
            (F.coalesce("ind_registro_deduplicado", F.lit(False)) == F.col("_c")), "left")
        .withColumn("sk_pago", F.coalesce("sk_pago", F.lit(SK_NO_APLICA)))
        .drop("_m", "_a", "_b", "_c"))

def sk_fecha(col):
    """La llave de fecha es AAAAMMDD; nula se mapea al centinela."""
    return F.coalesce(F.date_format(col, "yyyyMMdd").cast("int"), F.lit(SK_NO_APLICA))

## hecho_compra

Grano: una compra de vehículo. 500 filas.

**`fecha_ingreso_concesionario` no se carga.** Se verificó que coincide al 100% con
la fecha de la transacción de compra en las 500 placas, así que es redundante.
Cargarla duplicaría el mismo dato en dos lugares, lo que es una fuente de
inconsistencia futura.

In [0]:
compra = sv.filter(F.col("tipo_transaccion") == "COMPRA")
compra = sk_vehiculo_vigente(compra)
compra = sk_simple(compra, dim_sucursal, "nombre_sucursal", "nombre_sucursal", "sk_sucursal")
compra = sk_simple(compra, dim_proveedor, "nombre_proveedor", "nombre_proveedor", "sk_proveedor")
compra = sk_pago(compra)

(compra.select(
    sk_fecha("fecha_inicio").alias("sk_fecha_compra"),
    "sk_vehiculo", "sk_sucursal", "sk_proveedor", "sk_pago",
    F.col("id_transaccion"),
    F.col("valor_total").alias("valor_compra"),
    F.col("km_transaccion").alias("km_al_ingreso"))
 .write.mode("overwrite").saveAsTable(f"{CATALOGO}.gold.hecho_compra"))

## hecho_alquiler

Grano: un contrato de alquiler. 4356 filas.

**Se cargan `valor_alquiler` y `valor_teorico` juntos.** Esta es la materialización
de la decisión de silver de marcar y no corregir el ~10% de montos inconsistentes.
El tablero puede mostrar ingreso facturado e ingreso teórico lado a lado, y la
brecha se convierte en un hallazgo cuantificado para VehiAlpes en lugar de un ajuste
que nadie ve.

**`sk_cliente` apunta al registro capturado, no al unificado.** El hecho guarda la
llave tal como se capturó; la navegación a la persona consolidada se hace dentro de
la dimensión vía `sk_cliente_unificado`. Es la mecánica del SCD tipo 7 y preserva
ambas lecturas.

In [0]:
alquiler = sv.filter(F.col("tipo_transaccion") == "ALQUILER")
alquiler = sk_vehiculo_vigente(alquiler)
alquiler = sk_simple(alquiler, dim_cliente, "id_cliente", "id_cliente", "sk_cliente")
alquiler = sk_simple(alquiler, dim_sucursal, "nombre_sucursal", "nombre_sucursal", "sk_sucursal")
alquiler = sk_pago(alquiler)

(alquiler.select(
    sk_fecha("fecha_inicio").alias("sk_fecha_entrega"),
    sk_fecha("fecha_fin").alias("sk_fecha_devolucion"),
    "sk_vehiculo", "sk_cliente", "sk_sucursal", "sk_pago",
    F.col("id_transaccion"),
    F.col("valor_total").alias("valor_alquiler"),
    F.col("valor_teorico").alias("valor_alquiler_teorico"),
    F.col("valor_seguro").alias("valor_seguro_cobrado"),
    "dias_alquiler", "kms_recorridos",
    F.col("km_transaccion").alias("km_al_inicio"))
 .write.mode("overwrite").saveAsTable(f"{CATALOGO}.gold.hecho_alquiler"))

## hecho_venta

Grano: una venta. 150 filas.

**Nota sobre una ambigüedad no resuelta.** De las 9 placas con venta duplicada, la
placa WMJ514 trae dos registros idénticos salvo el cliente (898 y 1203). La regla de
deduplicación por `id_transaccion` menor escoge uno, pero es una elección arbitraria
entre dos alternativas igualmente plausibles. Se documenta como pregunta abierta
para VehiAlpes en lugar de presentarla como resuelta.

In [0]:
venta = sv.filter(F.col("tipo_transaccion") == "VENTA")
venta = sk_vehiculo_vigente(venta)
venta = sk_simple(venta, dim_cliente, "id_cliente", "id_cliente", "sk_cliente")
venta = sk_simple(venta, dim_sucursal, "nombre_sucursal", "nombre_sucursal", "sk_sucursal")
venta = sk_pago(venta)

(venta.select(
    sk_fecha("fecha_inicio").alias("sk_fecha_venta"),
    "sk_vehiculo", "sk_cliente", "sk_sucursal", "sk_pago",
    F.col("id_transaccion"),
    F.col("valor_total").alias("valor_venta"),
    F.col("km_transaccion").alias("km_al_momento_venta"))
 .write.mode("overwrite").saveAsTable(f"{CATALOGO}.gold.hecho_venta"))

## hecho_ocupacion_diaria (snapshot periódico)

Grano: un vehículo por día, desde la compra hasta la venta o hasta el horizonte.
~470.781 filas.

### Decisión: intervalo semiabierto

Un vehículo se marca ocupado desde `fecha_inicio` hasta `fecha_fin - 1`, no hasta
`fecha_fin`. La razón es que el valor del alquiler se calcula como
`(fecha_fin - fecha_inicio) × tarifa`, luego **el día de devolución no se cobra**.

El beneficio de este criterio es que produce una prueba de conciliación exacta: la
suma de `esta_alquilado` en este snapshot debe igualar la suma de `dias_alquiler` en
el hecho transaccional. Se verifica al final del notebook.

### Decisión: sin dimensión de cliente

Un día de ocupación no es atribuible a un solo cliente cuando hay alquileres
solapados (se detectaron 7 casos reales, en 7 placas distintas). Agregar esa FK obligaría a duplicar filas
y rompería la semi-aditividad. Para cruzar ocupación con cliente se hace
drill-across contra `hecho_alquiler`.

### Decisión: sí se particiona, a diferencia del resto

Es el único hecho que se particiona (por año). Con ~471 mil filas frente a ~5 mil
en el resto del modelo, la poda de particiones es lo que evita escanear todo el
histórico en cada consulta del tablero.

In [0]:
hc = spark.table(f"{CATALOGO}.gold.hecho_compra")
hv = spark.table(f"{CATALOGO}.gold.hecho_venta")
ha = spark.table(f"{CATALOGO}.gold.hecho_alquiler")
dim_fecha = spark.table(f"{CATALOGO}.gold.dim_fecha").filter(F.col("sk_fecha") > 0)

HORIZONTE = sv.select(F.greatest(F.max("fecha_inicio"), F.max("fecha_fin"))).first()[0]

# Ventana de vida de cada placa: de la compra a la venta, o al horizonte si sigue en flota
vida = (hc.join(dim_vehiculo.select("sk_vehiculo", "placa"), "sk_vehiculo")
    .select("placa", F.col("sk_fecha_compra"), "sk_sucursal")
    .join(dim_fecha.select(F.col("sk_fecha").alias("sk_fecha_compra"),
                           F.col("fecha").alias("fecha_compra")), "sk_fecha_compra")
    .join(hv.join(dim_vehiculo.select("sk_vehiculo", "placa"), "sk_vehiculo")
            .join(dim_fecha.select(F.col("sk_fecha").alias("sk_fecha_venta"),
                                   F.col("fecha").alias("fecha_venta")), "sk_fecha_venta")
            .select("placa", "fecha_venta"), "placa", "left")
    .withColumn("fecha_salida", F.coalesce("fecha_venta", F.lit(HORIZONTE))))

# Una fila por vehículo-día
calendario = (vida
    .withColumn("fecha", F.explode(F.sequence(
        F.col("fecha_compra"), F.col("fecha_salida"), F.expr("interval 1 day"))))
    .select("placa", "fecha", "sk_sucursal", "fecha_compra"))

# Intervalos ocupados, con el criterio semiabierto
ocupados = (ha.join(dim_vehiculo.select("sk_vehiculo", "placa"), "sk_vehiculo")
    .join(dim_fecha.select(F.col("sk_fecha").alias("sk_fecha_entrega"),
                           F.col("fecha").alias("f_ini")), "sk_fecha_entrega")
    .join(dim_fecha.select(F.col("sk_fecha").alias("sk_fecha_devolucion"),
                           F.col("fecha").alias("f_fin")), "sk_fecha_devolucion")
    .filter(F.col("dias_alquiler") > 0)
    .withColumn("fecha", F.explode(F.sequence(
        F.col("f_ini"), F.date_sub(F.col("f_fin"), 1), F.expr("interval 1 day"))))
    .select("placa", "fecha").distinct()
    .withColumn("_ocupado", F.lit(1)))

ocupacion = (calendario.join(ocupados, ["placa", "fecha"], "left")
    .withColumn("esta_alquilado", F.coalesce("_ocupado", F.lit(0)))
    .withColumn("esta_disponible", F.lit(1) - F.col("esta_alquilado"))
    .withColumn("dias_desde_ingreso", F.datediff("fecha", "fecha_compra"))
    .withColumn("anio", F.year("fecha")))

# La versión del vehículo vigente ese día determina la tarifa
ocupacion = (ocupacion
    .join(dim_vehiculo.filter(F.col("sk_vehiculo") > 0).select(
            "sk_vehiculo", F.col("placa").alias("_p"),
            F.col("fe_vig_ini").alias("_ini"), F.col("fe_vig_fin").alias("_fin"),
            F.col("costo_alquiler_dia").alias("tarifa_vigente_dia")),
          (F.col("placa") == F.col("_p")) &
          (F.col("fecha") >= F.col("_ini")) & (F.col("fecha") <= F.col("_fin")), "left")
    .withColumn("sk_vehiculo", F.coalesce("sk_vehiculo", F.lit(SK_NO_APLICA))))

(ocupacion.select(
    sk_fecha("fecha").alias("sk_fecha"),
    "sk_vehiculo", "sk_sucursal",
    "esta_alquilado", "esta_disponible", "dias_desde_ingreso", "tarifa_vigente_dia",
    "anio")
 .write.mode("overwrite").partitionBy("anio")
 .saveAsTable(f"{CATALOGO}.gold.hecho_ocupacion_diaria"))

## hecho_ciclo_vida (snapshot acumulativo)

Grano: una placa. 500 filas.

### Decisión: `MERGE` en lugar de `overwrite`

Un snapshot acumulativo se **actualiza** a medida que el vehículo avanza por sus
hitos: cuando una placa se vende, hay que rellenar `sk_fecha_venta` y `valor_venta`
en una fila que ya existe. Esto tensiona la naturaleza de solo-anexado de Delta.

Se implementa con `MERGE INTO`, que es la primitiva de Delta para esta situación.
En la primera ejecución equivale a un insert masivo; en las siguientes actualiza
solo las filas cuyos hitos cambiaron.

### Advertencia analítica que hay que documentar

350 de 500 vehículos siguen en flota, así que `valor_venta` y todo lo que dependa
del cierre del ciclo está vacío en el 70% de las filas. Cualquier promedio sobre
esas medidas sufre **sesgo de supervivencia**: se calcula solo sobre los vehículos
que ya salieron. Por eso se incluye `esta_vendido`, para que el tablero pueda
filtrar explícitamente y el usuario sepa sobre qué población está midiendo.

In [0]:
placas = dim_vehiculo.filter(F.col("sk_vehiculo") > 0).select("sk_vehiculo", "placa")
f_cal = dim_fecha.select(F.col("sk_fecha"), F.col("fecha"))

compra_x = (hc.join(placas, "sk_vehiculo")
    .join(f_cal.withColumnRenamed("sk_fecha", "sk_fecha_compra")
               .withColumnRenamed("fecha", "fecha_compra"), "sk_fecha_compra")
    .select("placa", "sk_fecha_compra", "fecha_compra", "valor_compra",
            "sk_proveedor", "km_al_ingreso"))

venta_x = (hv.join(placas, "sk_vehiculo")
    .join(f_cal.withColumnRenamed("sk_fecha", "sk_fecha_venta")
               .withColumnRenamed("fecha", "fecha_venta"), "sk_fecha_venta")
    .select("placa", "sk_fecha_venta", "fecha_venta", "valor_venta",
            F.col("sk_cliente").alias("sk_cliente_comprador")))

alq_x = (ha.join(placas, "sk_vehiculo")
    .join(f_cal.withColumnRenamed("sk_fecha", "sk_fecha_entrega")
               .withColumnRenamed("fecha", "fecha_entrega"), "sk_fecha_entrega")
    .groupBy("placa").agg(
        F.min("fecha_entrega").alias("fecha_primer_alquiler"),
        F.max("fecha_entrega").alias("fecha_ultimo_alquiler"),
        F.sum("valor_alquiler").alias("ingreso_total_alquileres"),
        F.count("*").alias("num_alquileres"),
        F.sum("dias_alquiler").alias("dias_totales_alquilado"),
        F.sum("kms_recorridos").alias("kms_totales_alquiler")))

ciclo = (compra_x
    .join(alq_x, "placa", "left")
    .join(venta_x, "placa", "left")
    .join(dim_vehiculo.filter(F.col("es_version_actual"))
            .select("placa", F.col("sk_vehiculo").alias("sk_vehiculo_actual")), "placa")
    .withColumn("esta_vendido", F.col("sk_fecha_venta").isNotNull())
    .withColumn("dias_en_flota", F.datediff(
        F.coalesce("fecha_venta", F.lit(HORIZONTE)), F.col("fecha_compra")))
    .withColumn("lag_compra_primer_alquiler",
        F.datediff("fecha_primer_alquiler", "fecha_compra"))
    .withColumn("lag_ultimo_alquiler_venta",
        F.datediff("fecha_venta", "fecha_ultimo_alquiler"))
    .select(
        F.col("sk_vehiculo_actual").alias("sk_vehiculo"),
        "sk_fecha_compra",
        sk_fecha("fecha_primer_alquiler").alias("sk_fecha_primer_alquiler"),
        sk_fecha("fecha_ultimo_alquiler").alias("sk_fecha_ultimo_alquiler"),
        F.coalesce("sk_fecha_venta", F.lit(SK_NO_APLICA)).alias("sk_fecha_venta"),
        "sk_proveedor",
        F.coalesce("sk_cliente_comprador", F.lit(SK_NO_APLICA)).alias("sk_cliente_comprador"),
        "valor_compra", "ingreso_total_alquileres", "valor_venta",
        F.coalesce("num_alquileres", F.lit(0)).alias("num_alquileres"),
        F.coalesce("dias_totales_alquilado", F.lit(0)).alias("dias_totales_alquilado"),
        F.coalesce("kms_totales_alquiler", F.lit(0)).alias("kms_totales_alquiler"),
        "dias_en_flota", "lag_compra_primer_alquiler", "lag_ultimo_alquiler_venta",
        "esta_vendido"))

ciclo.createOrReplaceTempView("v_ciclo_nuevo")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOGO}.gold.hecho_ciclo_vida
    AS SELECT * FROM v_ciclo_nuevo WHERE 1 = 0""")

spark.sql(f"""
    MERGE INTO {CATALOGO}.gold.hecho_ciclo_vida AS destino
    USING v_ciclo_nuevo AS origen
      ON destino.sk_vehiculo = origen.sk_vehiculo
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## hecho_historia_vehiculo (factless)

Grano: un cambio de versión del vehículo. 879 filas.

Tabla de hechos sin medidas propias: `conteo` es constante en 1 y existe para
contar eventos, siguiendo el patrón visto en clase. Los deltas sí son medidas
reales, calculadas contra la versión anterior de la misma placa.

Habilita preguntas que ningún otro hecho responde: cuántos vehículos ajustaron
tarifa en un trimestre y con qué magnitud. En particular, deja visible el evento más
notable de los datos: la actualización masiva del 1 de noviembre de 2024, que elevó
el seguro de 45.000 a 52.000 en las 500 placas simultáneamente.

In [0]:
por_placa = W.partitionBy("placa").orderBy("fe_vig_ini")

historia = (dim_vehiculo.filter(F.col("sk_vehiculo") > 0)
    .withColumn("_tarifa_ant", F.lag("costo_alquiler_dia").over(por_placa))
    .withColumn("_seguro_ant", F.lag("valor_seguro").over(por_placa))
    .withColumn("_color_ant", F.lag("color").over(por_placa))
    .filter(F.col("num_version") > 1)
    .select(
        "sk_vehiculo",
        sk_fecha("fe_vig_ini").alias("sk_fecha_inicio_vigencia"),
        (F.col("costo_alquiler_dia") - F.col("_tarifa_ant")).alias("delta_costo_alquiler_dia"),
        (F.col("valor_seguro") - F.col("_seguro_ant")).alias("delta_valor_seguro"),
        (~F.col("color").eqNullSafe(F.col("_color_ant"))).alias("cambio_color"),
        (F.col("costo_alquiler_dia") != F.col("_tarifa_ant")).alias("cambio_tarifa"),
        (F.col("valor_seguro") != F.col("_seguro_ant")).alias("cambio_seguro"),
        F.lit(1).alias("conteo")))

historia.write.mode("overwrite") \
    .saveAsTable(f"{CATALOGO}.gold.hecho_historia_vehiculo")

## Validación de la capa

### La prueba que realmente importa

La conciliación entre el snapshot diario y el hecho transaccional es la validación
más valiosa del pipeline: si el criterio de intervalo semiabierto está bien
implementado, los dos totales coinciden exactamente. Si no coinciden, hay un error
de frontera en el snapshot y todas las métricas de ocupación son falsas.

In [0]:
# Conteos por hecho
for hecho in ["hecho_compra", "hecho_alquiler", "hecho_venta",
              "hecho_ocupacion_diaria", "hecho_ciclo_vida", "hecho_historia_vehiculo"]:
    print(f"{hecho}: {spark.table(f'{CATALOGO}.gold.{hecho}').count():,} filas")

hecho_compra: 500 filas
hecho_alquiler: 4,356 filas
hecho_venta: 150 filas
hecho_ocupacion_diaria: 470,781 filas
hecho_ciclo_vida: 500 filas
hecho_historia_vehiculo: 879 filas


In [0]:
# Conciliación de días ocupados entre granos
dias_snapshot = spark.table(f"{CATALOGO}.gold.hecho_ocupacion_diaria") \
    .agg(F.sum("esta_alquilado")).first()[0]
dias_transaccional = spark.table(f"{CATALOGO}.gold.hecho_alquiler") \
    .filter(F.col("dias_alquiler") > 0).agg(F.sum("dias_alquiler")).first()[0]

print(f"Días-carro en el snapshot diario : {dias_snapshot:,}")
print(f"Días-carro en el hecho de alquiler: {dias_transaccional:,}")
print(f"Diferencia: {dias_transaccional - dias_snapshot:,}")

Días-carro en el snapshot diario : 37,345
Días-carro en el hecho de alquiler: 37,410
Diferencia: 65


La diferencia no es cero, y **debe** no serlo: 7 contratos de una misma placa se
solapan en el tiempo. En el snapshot un día-carro cuenta una sola vez porque la
bandera es booleana, mientras que en el hecho transaccional los días de los dos
contratos se suman por separado.

Lo valioso es que la discrepancia es **predecible al día**: debe igualar exactamente
la cantidad de días-carro cubiertos por más de un contrato. Sobre esta muestra son
65 días (37.410 transaccionales frente a 37.345 en el snapshot). La celda siguiente
lo verifica como aserción en lugar de dejarlo a interpretación: si la diferencia no
coincide con los solapamientos, hay un error real de fronteras en el snapshot y las
métricas de ocupación no son confiables.

In [0]:
# Días-carro cubiertos por más de un contrato: explica la diferencia exacta
dias_solapados = (ha.filter(F.col("dias_alquiler") > 0)
    .join(dim_fecha.select(F.col("sk_fecha").alias("sk_fecha_entrega"),
                           F.col("fecha").alias("f_ini")), "sk_fecha_entrega")
    .join(dim_fecha.select(F.col("sk_fecha").alias("sk_fecha_devolucion"),
                           F.col("fecha").alias("f_fin")), "sk_fecha_devolucion")
    .withColumn("fecha", F.explode(F.sequence(
        F.col("f_ini"), F.date_sub(F.col("f_fin"), 1), F.expr("interval 1 day"))))
    .groupBy("sk_vehiculo", "fecha").count()
    .filter(F.col("count") > 1)
    .agg(F.sum(F.col("count") - 1)).first()[0] or 0)

print(f"Días-carro cubiertos por más de un contrato: {dias_solapados:,}")
assert dias_transaccional - dias_snapshot == dias_solapados, (
    f"La diferencia ({dias_transaccional - dias_snapshot}) no coincide con los "
    f"solapamientos ({dias_solapados}): hay un error de fronteras en el snapshot")
print("Conciliación exacta entre el snapshot diario y el hecho transaccional")

Días-carro cubiertos por más de un contrato: 65
Conciliación exacta entre el snapshot diario y el hecho transaccional


In [0]:
# Ninguna llave foránea nula en ningún hecho
FK_POR_HECHO = {
    "hecho_compra": ["sk_fecha_compra", "sk_vehiculo", "sk_sucursal", "sk_proveedor", "sk_pago"],
    "hecho_alquiler": ["sk_fecha_entrega", "sk_fecha_devolucion", "sk_vehiculo",
                       "sk_cliente", "sk_sucursal", "sk_pago"],
    "hecho_venta": ["sk_fecha_venta", "sk_vehiculo", "sk_cliente", "sk_sucursal", "sk_pago"],
    "hecho_ocupacion_diaria": ["sk_fecha", "sk_vehiculo", "sk_sucursal"],
    "hecho_ciclo_vida": ["sk_vehiculo", "sk_fecha_compra", "sk_fecha_venta"],
    "hecho_historia_vehiculo": ["sk_vehiculo", "sk_fecha_inicio_vigencia"],
}

for hecho, llaves in FK_POR_HECHO.items():
    df = spark.table(f"{CATALOGO}.gold.{hecho}")
    for llave in llaves:
        nulas = df.filter(F.col(llave).isNull()).count()
        assert nulas == 0, f"{hecho}.{llave} tiene {nulas} valores nulos"
print("Ninguna llave foránea nula en los seis hechos")

Ninguna llave foránea nula en los seis hechos


In [0]:
# Utilización de flota — la métrica que habilita el snapshot
spark.sql(f"""
    SELECT f.anio,
           SUM(o.esta_alquilado) AS dias_alquilados,
           COUNT(*) AS dias_carro_disponibles,
           ROUND(100.0 * SUM(o.esta_alquilado) / COUNT(*), 1) AS pct_ocupacion
    FROM {CATALOGO}.gold.hecho_ocupacion_diaria o
    JOIN {CATALOGO}.gold.dim_fecha f ON o.sk_fecha = f.sk_fecha
    GROUP BY f.anio ORDER BY f.anio""").show()

+----+---------------+----------------------+-------------+
|anio|dias_alquilados|dias_carro_disponibles|pct_ocupacion|
+----+---------------+----------------------+-------------+
|2022|              0|                    33|          0.0|
|2023|              0|                 56380|          0.0|
|2024|          26620|                174169|         15.3|
|2025|          10533|                160112|          6.6|
|2026|            192|                 80087|          0.2|
+----+---------------+----------------------+-------------+



**Cómo leer este resultado.** La ocupación cae de 15,3% en 2024 a 6,6% en 2025 y a
0,2% en 2026. Esa caída **no es una tendencia del negocio**: la muestra
contiene 3137 alquileres en 2024, 1230 en 2025 y solo 23 en 2026, así que la serie
refleja el sesgo del muestreo. El modelo habilita correctamente la métrica; la
advertencia sobre la representatividad de los datos debe acompañar cualquier
presentación de esta cifra.

In [0]:
for hecho in FK_POR_HECHO:
    spark.sql(f"OPTIMIZE {CATALOGO}.gold.{hecho}")